In [41]:
import pandas as pd
import glob

In [42]:
path = 'data/'

In [43]:
prod_files = glob.glob(f"{path}/*Produktionsplanung.xlsx")
prod_dfs = []

In [44]:
for file in prod_files:
    df = pd.read_excel(file)
    
    # leave only the necessary columns (the names are in German and in all files are the same)
    df_reduced = df[['Prod.Dat.', 'Gebinde', 'Bezeichnung']]
    
    # remame for convenience
    df_reduced.columns = ['date', 'gebinde', 'dish_name']
    
    # adding it to the dataframelist
    prod_dfs.append(df_reduced)

In [45]:
prod_full_df = pd.concat(prod_dfs, ignore_index=True)

In [46]:
prod_full_df.head(10)

,date,gebinde,dish_name
0,2014-01-07,Reis,Butterreis
1,2014-01-07,Kartoffeln,"Kartoffeln, geschält"
2,2014-01-07,Kartoffelsalat,Kartoffelsalat fertig
3,2014-01-07,Nudeln,Nudeln Fusilli
4,2014-01-07,Spätzle,"Spätzle, frisch"
5,2014-01-07,Essen 1 Mensen VK,Hacklett - Hacksteak
6,2014-01-07,Essen 2 Mensen VK,Puten Cordon Bleu mit Zitrone
7,2014-01-07,Essen 3 Mensen VK,Pasta mit Tomate-Gemüsesoße
8,2014-01-07,Pommes frites,Pommes-Frites
9,2014-01-08,Reis,Butterreis


In [47]:
prod_full_df['date'] = pd.to_datetime(prod_full_df['date'], format='%d/%m/%Y')

In [48]:
prod_full_df.head(10)

,date,gebinde,dish_name
0,2014-01-07,Reis,Butterreis
1,2014-01-07,Kartoffeln,"Kartoffeln, geschält"
2,2014-01-07,Kartoffelsalat,Kartoffelsalat fertig
3,2014-01-07,Nudeln,Nudeln Fusilli
4,2014-01-07,Spätzle,"Spätzle, frisch"
5,2014-01-07,Essen 1 Mensen VK,Hacklett - Hacksteak
6,2014-01-07,Essen 2 Mensen VK,Puten Cordon Bleu mit Zitrone
7,2014-01-07,Essen 3 Mensen VK,Pasta mit Tomate-Gemüsesoße
8,2014-01-07,Pommes frites,Pommes-Frites
9,2014-01-08,Reis,Butterreis


In [65]:
verkaufs_files = glob.glob(f"{path}/*Verkaufszahlen.xlsx")  # Папку укажи свою при необходимости

all_records = []

for file in verkaufs_files:
    df_raw = pd.read_excel(file, header=None)

    # Названия блюд: обычные + Essen
    dish_names = list(df_raw.loc[5:19, 6]) + list(df_raw.loc[21:30, 6])
    
    # Даты в строке 4 (индекс 3), начиная с колонки I (индекс 8)
    dates = list(df_raw.loc[3, 8:])
    
    # Сохраняем все записи из текущего файла
    for dish_row_idx, dish_name in zip(list(range(5, 20)) + list(range(21, 31)), dish_names):
        for date_col_idx, date in enumerate(dates, start=8):
            sold = df_raw.loc[dish_row_idx, date_col_idx]
            if pd.notna(sold):
                if 'Gesamt' not in str(date):  # Убираем "Gesamt" столбцы
                    all_records.append({
                        'dish_name': dish_name,
                        'date': date,
                        'sold_amount': sold
                    })

# Превращаем всё в единый датафрейм
verkauf_df_all = pd.DataFrame(all_records)


In [66]:
verkauf_df_all.head()

,dish_name,date,sold_amount
0,Beilage Kartoffeln,2014.01.27,16
1,Beilage Kartoffeln,2014.05.05,25
2,Beilage Kartoffeln,2014.11.17,2
3,Gemüsereis,2014.04.08,5
4,Gemüsereis,2014.05.08,49


In [68]:
verkauf_df_all['date'] = pd.to_datetime(verkauf_df_all['date'], format='%Y.%m.%d')
verkauf_df_all = verkauf_df_all.sort_values('date').reset_index(drop=True)
verkauf_df_all.head()

,dish_name,date,sold_amount
0,Kartoffelsalat,2014-01-07,74
1,Essen 1,2014-01-07,159
2,Essen 3,2014-01-07,185
3,Essen- Soziales ohne Berechnung,2014-01-07,1
4,Kartoffeln,2014-01-07,3


In [69]:
verkauf_df_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16134 entries, 0 to 16133
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   dish_name    11891 non-null  object        
 1   date         16134 non-null  datetime64[ns]
 2   sold_amount  16134 non-null  int64         
dtypes: datetime64[ns](1), int64(1), object(1)
memory usage: 378.3+ KB


In [70]:
prod_full_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16254 entries, 0 to 16253
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       16254 non-null  datetime64[ns]
 1   gebinde    15810 non-null  object        
 2   dish_name  16254 non-null  object        
dtypes: datetime64[ns](1), object(2)
memory usage: 381.1+ KB
